##PARAMETROS

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_finalproject")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

##CARGA EN DIMENSIONES

In [0]:
# =========================
# LOAD SILVER
# =========================
customers_df = spark.table("catalog_finalproject.silver.customers_sz")
products_df = spark.table("catalog_finalproject.silver.products_sz")
orders_df = spark.table("catalog_finalproject.silver.orders_sz")
sales_df = spark.table("catalog_finalproject.silver.sales_sz")

In [0]:

# =========================
# DIM DATE basado en las ordenes de compra
# =========================
dim_date = (
    orders_df
    .select("order_date")
    .distinct()
    .select(
        col("order_date").alias("date"),
        year("order_date").alias("year"),
        month("order_date").alias("month"),
        dayofmonth("order_date").alias("day"),
        date_format("order_date", "yyyy-MM").alias("year_month")
    )
)
dim_date.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.dim_date")


In [0]:

# =========================
# DIM CUSTOMER
# =========================
dim_customer = customers_df.select(
    "customer_id",
    "name",
    "email",
    "country",
    "city",
    "created_date"
)
dim_customer.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.dim_customer")

In [0]:
# =========================
# DIM PRODUCT
# =========================
dim_product = products_df.select(
    "product_id",
    "product_name",
    "category",
    "price",
    "availability"
)

dim_product.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.dim_product")


In [0]:

# =========================
# DIM CHANNEL / BRANCH sucursal
# =========================
dim_channel = (
    orders_df
    .select("channel", "branch_name", "state")
    .distinct()
)
dim_channel.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.dim_channel")


##CARGA EN FACT SALES

In [0]:
# =========================
# FACT SALES 
# =========================
fact_sales = (
    sales_df
    .join(orders_df, "order_id", "inner")
    .select(
        col("sale_id"),
        col("order_id"),
        col("customer_id"),
        col("product_id"),
        col("order_date"),
        col("channel"),
        col("branch_name"),
        col("state"),
        col("quantity"),
        col("price").alias("unit_price"),
        col("total_amount")
    )
)

# =========================
# COLUMNAS DE VALOR como una simulacion de descuento del 5% y un 15% de iva
# =========================

fact_sales = (
    fact_sales
    # % descuento simulado (puedes cambiar lógica luego)
    .withColumn("discount_pct", lit(0.05))  # 5%
    # monto con descuento
    .withColumn("net_amount", col("total_amount") * (1 - col("discount_pct")))
    # impuesto (15%)
    .withColumn("tax_amount", col("net_amount") * lit(0.12))
    # total final
    .withColumn("gross_amount", col("net_amount") + col("tax_amount"))
    # flag de ticket alto (ejemplo > 500)
    .withColumn("high_ticket_flag", (col("gross_amount") > 500).cast("int"))
)
fact_sales.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.fact_sales")
